# 🛰️ TKG Solar — Data Pipeline (M1) Walkthrough

Đọc dữ liệu thật **OPSD + NSRDB + Himawari-8** và đi qua **từng bước** của data
pipeline (`src/data_pipeline/`), mỗi bước phân tách bằng đường kẻ rõ ràng kèm
**công thức** toán học tương ứng.

> ⚠️ **Phạm vi tái lập (đọc trước).** 3 nguồn dữ liệu *không cùng vị trí địa lý*
> (OPSD PV = châu Âu; NSRDB/Himawari = châu Á). Theo quyết định đã-được-thông-báo
> của người dùng, dự án **trung thành với lựa chọn dữ liệu của paper** → đây là tái
> lập **cơ học/mã nguồn**, không phải mô hình hợp lệ vật lý. "Aligned" = canh theo
> đồng hồ UTC, *không phải* ghép cặp vật lý. Xem `docs/assumptions.md`.

**Luồng M1:**
```
raw sources ─▶ align (10-min grid) ─▶ clean ─▶ split ─▶ clip ─▶ scale ─▶ window ─▶ DataLoader
```

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 0 · Thiết lập môi trường

Thêm project root vào `sys.path`, nạp `Config` và các hằng số *shape contract*
(`src/common/shapes.py` — nguồn chân lý duy nhất cho mọi chiều tensor).

In [ ]:
import sys, pathlib
# Notebook nằm trong notebooks/ -> project root là thư mục cha.
ROOT = pathlib.Path.cwd()
if (ROOT / 'src').exists() is False:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import os; os.chdir(ROOT)

import numpy as np, pandas as pd
import matplotlib.pyplot as plt

from src.training.config import Config
from src.common.shapes import (
    METEO_FEATURES, N_METEO_FEATURES, SAT_CHANNELS, SAT_IMG_SIZE,
    HORIZON_MINUTES, HORIZON_STEPS, BASE_CADENCE_MIN, N_HORIZONS,
)

cfg = Config.from_yaml('configs/vn_real_2016.yaml')
print('Project root :', ROOT)
print('Cadence      :', BASE_CADENCE_MIN, 'min')
print('Horizons     :', HORIZON_MINUTES, 'min  ->  steps', HORIZON_STEPS)
print('Meteo feats  :', N_METEO_FEATURES, '=', METEO_FEATURES)
print('Sat tensor   :', f'C={SAT_CHANNELS}, H=W={cfg.img_size}')
print('Window k     :', cfg.k, '| split', cfg.train_frac, cfg.val_frac)

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 1 · Đọc dữ liệu thô (load raw sources)

Mỗi nguồn được đọc trên **nhịp lấy mẫu gốc** của nó với chỉ mục thời gian UTC
(`src/data_pipeline/loaders.py`):

| Nguồn | Hàm | Nhịp gốc | Output |
|---|---|---|---|
| OPSD PV | `load_opsd` | 15 phút | `Series` công suất PV $[T]$ |
| NSRDB meteo | `load_nsrdb` | 30/60 phút | `DataFrame` $[T, 7]$ |
| Himawari sat | `load_himawari` | 10 phút | `ndarray` $[T, C, H, W]$ |

Tập đặc trưng khí tượng (thứ tự **cố định**, dùng chung loader↔scaler↔encoder):

$$\mathbf{m}_t = [\text{ghi},\,\text{dni},\,\text{dhi},\,T_{air},\,RH,\,v_{wind},\,p_{surf}]^\top \in \mathbb{R}^{7}$$

In [ ]:
from src.data_pipeline.loaders import (
    load_opsd, load_nsrdb, load_himawari, find_himawari_file,
)

pv_raw    = load_opsd(cfg.opsd_path)
meteo_raw = load_nsrdb(cfg.nsrdb_path)
him_file  = find_himawari_file(cfg.himawari_dir)
sat_raw, sat_ts = load_himawari(him_file)

def span(idx):
    return f'{idx.min()}  ..  {idx.max()}'

print(f'OPSD  PV   : {pv_raw.shape}      {span(pv_raw.index)}')
print(f'NSRDB meteo: {meteo_raw.shape}   {span(meteo_raw.index)}')
print(f'Himawari   : {sat_raw.shape}  {span(sat_ts)}')
print()
print('Mẫu khí tượng (5 hàng đầu):')
display(meteo_raw.head())

In [ ]:
# Trực quan: công suất PV thô (OPSD) + 1 frame vệ tinh + GHI khí tượng.
fig, ax = plt.subplots(1, 3, figsize=(15, 3.2))
pv_raw.iloc[:96*3].plot(ax=ax[0], lw=.8); ax[0].set_title('OPSD PV (3 ngày, nhịp 15-phút)')
ax[0].set_ylabel('PV power')
ax[1].imshow(sat_raw[len(sat_raw)//2, 0], cmap='gray'); ax[1].set_title('Himawari frame (B03)')
ax[1].axis('off')
meteo_raw['ghi'].iloc[:48*2].plot(ax=ax[2], lw=.9, color='orange'); ax[2].set_title('NSRDB GHI (2 ngày)')
plt.tight_layout(); plt.show()

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 2 · Canh thời gian → lưới chung 10 phút (time alignment)

`build_common_grid` (`time_alignment.py`) đưa cả 3 nguồn về **một lưới UTC 10 phút**
trên khoảng giao nhau, rồi giữ lại các mốc mà **cả ba** đều có dữ liệu.

**Khoảng giao thời gian:**
$$[t_0, t_1] = \Big[\max(\min_i \tau_i),\ \min(\max_i \tau_i)\Big],\quad i\in\{\text{pv, meteo, sat}\}$$

**Lưới đều bước** $\Delta=10$ phút:  $\;g_j = t_0 + j\,\Delta,\ j=0,1,\dots$

**Nội suy tuyến tính theo thời gian** (PV 15-phút & meteo → lưới 10-phút):
$$x(g) = x(\tau_a) + \frac{g-\tau_a}{\tau_b-\tau_a}\,\big(x(\tau_b)-x(\tau_a)\big),\quad \tau_a \le g \le \tau_b$$

**Vệ tinh** dùng *nearest-neighbour* với dung sai $\le \Delta$ (không nội suy ảnh).

**Cổng giao nhau (intersection gate):** chỉ giữ mốc $g_j$ thỏa
$$\text{mask}_j = \mathbb{1}[\text{pv}_j\ \text{notna}] \wedge \mathbb{1}[\text{meteo}_j\ \text{notna đủ 7}] \wedge \mathbb{1}[\exists\,\text{sat trong } \Delta]$$
và bắt buộc $\sum_j \text{mask}_j \ge \texttt{min\_steps}$ (nếu không → `EmptyOverlapError`).

In [ ]:
from src.data_pipeline.time_alignment import build_common_grid

grid = build_common_grid(
    pv_raw, meteo_raw, sat_raw, sat_ts,
    cadence_min=BASE_CADENCE_MIN, min_steps=cfg.min_steps, img_size=cfg.img_size,
)
ts = grid['timestamps']
print('Khoảng giao :', ts.min(), '..', ts.max())
print('Số bước chung T =', len(ts), f'(>= min_steps={cfg.min_steps})')
print('pv   :', grid['pv'].shape)
print('meteo:', grid['meteo'].shape)
print('sat  :', grid['sat'].shape, '<- đã resize về', cfg.img_size)

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 3 · Làm sạch (cleaning)

`clean_arrays` (`cleaning.py`) xử lý NaN còn sót sau nội suy và kẹp ảnh vệ tinh.

**Forward-fill rồi back-fill** theo trục thời gian cho pv/meteo:
$$\hat{x}_t = \begin{cases} x_t & x_t \text{ hợp lệ}\\ \hat{x}_{t-1} & \text{NaN (ffill)}\\ x_{t^*} & \text{NaN dẫn đầu, } t^*=\min\{t: x_t\ \text{hợp lệ}\}\ (\text{bfill}) \end{cases}$$

**Kẹp ảnh vệ tinh** về dải phản xạ hợp lệ (visible reflectance):
$$\text{sat}_t \leftarrow \operatorname{clip}\big(\text{nan\_to\_num}(\text{sat}_t),\,0,\,1\big)$$

> 📝 *Cắt outlier (z-score) KHÔNG làm ở đây* — nó cần thống kê chỉ-trên-train nên
> được hoãn tới **sau khi chia tập** (Bước 5) để tránh rò rỉ dữ liệu.

In [ ]:
from src.data_pipeline.cleaning import clean_arrays

nan_before = {k: int(np.isnan(grid[k]).sum()) for k in ('pv','meteo','sat')}
clean = clean_arrays(grid)
nan_after = {k: int(np.isnan(clean[k]).sum()) for k in ('pv','meteo','sat')}

print('NaN trước  :', nan_before)
print('NaN sau    :', nan_after)
print('sat range  : [', round(float(clean['sat'].min()),3), ',', round(float(clean['sat'].max()),3), ']')

sat, meteo, pv = clean['sat'], clean['meteo'], clean['pv']

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 4 · Chia tập theo thời gian (chronological split 70/15/15)

`chronological_bounds` (`splits.py`) — **không xáo trộn theo thời gian**; test là
giai đoạn mới nhất. Chia **trước khi** tạo cửa sổ → không có cửa sổ nào bắc qua
ranh giới tập (chống rò rỉ).

$$\text{train\_end} = \lfloor T\cdot 0.70 \rfloor,\qquad \text{val\_end} = \lfloor T\cdot(0.70+0.15) \rfloor$$
$$\text{train}=[0,\text{train\_end}),\ \ \text{val}=[\text{train\_end},\text{val\_end}),\ \ \text{test}=[\text{val\_end},T)$$

In [ ]:
from src.data_pipeline.splits import chronological_bounds

n = len(pv)
b = chronological_bounds(n, cfg.train_frac, cfg.val_frac)
print(f'T = {n}')
print(f'train [0:{b.train_end})        -> {b.train_end} bước')
print(f'val   [{b.train_end}:{b.val_end})      -> {b.val_end-b.train_end} bước')
print(f'test  [{b.val_end}:{n})      -> {n-b.val_end} bước')

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 5 · Cắt outlier — thống kê CHỈ-TRÊN-TRAIN (no leakage)

`fit_clip_bounds` / `apply_clip` (`cleaning.py`): tính ngưỡng $\pm z\sigma$ từ **tập
train**, rồi áp lên **cả ba** tập. Mặc định $z=5$.

$$\mu_f = \frac{1}{N_{tr}}\sum_{t\in\text{train}} x_{t,f},\qquad \sigma_f = \sqrt{\tfrac{1}{N_{tr}}\sum_{t\in\text{train}} (x_{t,f}-\mu_f)^2}$$
$$x_{t,f} \leftarrow \operatorname{clip}\big(x_{t,f},\ \mu_f - z\sigma_f,\ \mu_f + z\sigma_f\big),\quad z=5$$

In [ ]:
from src.data_pipeline.cleaning import fit_clip_bounds, apply_clip

m_lo, m_hi = fit_clip_bounds(meteo[b.train])
p_lo, p_hi = fit_clip_bounds(pv[b.train])

m_clipped = int((meteo != apply_clip(meteo, m_lo, m_hi)).sum())
p_clipped = int((pv != apply_clip(pv, p_lo, p_hi)).sum())
meteo = apply_clip(meteo, m_lo, m_hi)
pv    = apply_clip(pv,    p_lo, p_hi)

print('Ngưỡng meteo lo:', np.round(m_lo.ravel(), 2))
print('Ngưỡng meteo hi:', np.round(m_hi.ravel(), 2))
print(f'Giá trị bị cắt -> meteo: {m_clipped}, pv: {p_clipped}')

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 6 · Chuẩn hoá Min-Max — fit CHỈ-TRÊN-TRAIN

`fit_scalers` / `Scalers` (`scaling.py`). Hai scaler tách biệt để **inverse PV** lúc
đánh giá là rõ ràng; ảnh vệ tinh dùng min/max vô hướng (giả định gần $[0,1]$).

**Min-Max** (fit trên train, transform mọi tập):
$$x' = \frac{x - x_{\min}^{\text{train}}}{x_{\max}^{\text{train}} - x_{\min}^{\text{train}}}$$

**Inverse PV** (đưa dự báo về đơn vị gốc để tính MAE/RMSE/MAPE):
$$\hat{y} = y'\,(y_{\max}^{\text{train}} - y_{\min}^{\text{train}}) + y_{\min}^{\text{train}}$$

> 📝 Đỉnh test (về sau) có thể vượt max-train → giá trị scaled $>1$. **Không kẹp**
> (kẹp sẽ làm hỏng target); metric tính ở đơn vị gốc qua inverse-transform nên vẫn đúng.

In [ ]:
from src.data_pipeline.scaling import fit_scalers

scalers = fit_scalers(meteo[b.train], pv[b.train], sat[b.train])

meteo_s = scalers.transform_meteo(meteo)
pv_s    = scalers.transform_pv(pv)
sat_s   = scalers.transform_sat(sat)

print('PV  train range -> [%.3f, %.3f]' % (float(scalers.pv_scaler.data_min_[0]), float(scalers.pv_scaler.data_max_[0])))
print('sat train range -> [%.3f, %.3f]' % (scalers.sat_min, scalers.sat_max))
print('scaled pv_s     -> [%.3f, %.3f]  (test có thể >1)' % (pv_s.min(), pv_s.max()))
print('scaled meteo_s  ->', meteo_s.shape, ' scaled sat_s ->', sat_s.shape)
# Kiểm chứng inverse PV khôi phục đúng giá trị gốc:
recon = scalers.inverse_pv(pv_s[:5])
print('inverse PV OK   :', np.allclose(recon, pv[:5], atol=1e-4))

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 7 · Tạo cửa sổ trượt (sliding window — chỉ tính chỉ số)

`windowing.py` chỉ tính **chỉ số bắt đầu hợp lệ** (lazy — không nhân bản frame vệ
tinh $k$ lần). Dataset cắt mảng liên tục lúc `__getitem__`.

Cửa sổ bắt đầu tại $i$ dùng input bước $[i, i+k)$; "thời điểm hiện tại" là $i+k-1$;
target là PV tại $(i+k-1)+h$ cho mỗi $h \in$ `HORIZON_STEPS` $=(1,3,6)$.

**Chỉ số bắt đầu hợp lệ** (để input + mọi target nằm trong $[0,T)$):
$$i \in \{0, 1, \dots, T - k - \max_h h\}$$

**Chỉ số target tuyệt đối:**  $\;\text{tgt}(i) = \{\,i+k-1+h \;:\; h\in\{1,3,6\}\,\}$

In [ ]:
from src.data_pipeline.windowing import valid_starts, target_indices

starts_tr = valid_starts(b.train_end, cfg.k, HORIZON_STEPS)
print(f'k = {cfg.k}, max horizon step = {max(HORIZON_STEPS)}')
print(f'Số cửa sổ train = {len(starts_tr)}  (= train_len - k - max_h + 1)')
i0 = int(starts_tr[0])
print(f'Cửa sổ đầu i={i0}: input bước [{i0}, {i0+cfg.k}), hiện tại = {i0+cfg.k-1}')
print(f'  -> target tại {target_indices(i0, cfg.k, HORIZON_STEPS)}  (=+10/+30/+60 phút)')

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 8 · Dataset + DataLoader (một batch thực tế)

`SolarWindowDataset` (`dataset.py`) trả về mỗi mẫu một dict tensor; `DataLoader`
gộp thành batch. Hợp đồng shape mỗi mẫu:

$$\text{sat\_seq}\,[k,C,H,W],\quad \text{meteo\_seq}\,[k,7],\quad \text{pv\_hist}\,[k,1],\quad \text{target}\,[3]$$

In [ ]:
import torch
from torch.utils.data import DataLoader
from src.data_pipeline.dataset import SolarWindowDataset

train_ds = SolarWindowDataset(sat_s[b.train], meteo_s[b.train], pv_s[b.train], cfg.k, HORIZON_STEPS)
loader = DataLoader(train_ds, batch_size=cfg.batch_size, shuffle=False)
batch = next(iter(loader))
print('Số cửa sổ trong train_ds:', len(train_ds))
for key, val in batch.items():
    print(f'  {key:10s}: {tuple(val.shape)}  {val.dtype}')

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
## 9 · Toàn bộ M1 trong một lệnh (sanity check)

`DataPipeline.load` đóng gói **đúng** chuỗi 8 bước trên (load→align→clean→split→clip→
scale→window→loader) + cache bước align/clean tốn kém. Dưới đây xác nhận shape khớp.

In [ ]:
from src.data_pipeline import DataPipeline

splits = DataPipeline.load(
    cfg.opsd_path, cfg.nsrdb_path, cfg.himawari_dir,
    k=cfg.k, batch_size=cfg.batch_size, img_size=cfg.img_size,
    min_steps=cfg.min_steps, train_frac=cfg.train_frac, val_frac=cfg.val_frac,
)
print('meta:')
for kk, vv in splits.meta.items():
    print(f'  {kk:18s}: {vv}')
sb = next(iter(splits.train_loader))
print('\nbatch shapes:', {k: tuple(v.shape) for k, v in sb.items()})

<hr style="border:none;border-top:3px solid #2c7fb8;margin:8px 0">
### ✅ Tóm tắt luồng

| Bước | Module | Phép biến đổi chính |
|---|---|---|
| 1 Load | `loaders.py` | đọc 3 nguồn theo nhịp gốc, index UTC |
| 2 Align | `time_alignment.py` | lưới 10-phút + nội suy + cổng giao nhau |
| 3 Clean | `cleaning.py` | ffill/bfill NaN, kẹp sat $[0,1]$ |
| 4 Split | `splits.py` | 70/15/15 theo thời gian |
| 5 Clip | `cleaning.py` | $\mu\pm5\sigma$ (train-only) |
| 6 Scale | `scaling.py` | Min-Max (train-only) + inverse PV |
| 7 Window | `windowing.py` | chỉ số bắt đầu hợp lệ (lazy) |
| 8 Dataset | `dataset.py` | cắt cửa sổ → tensor dict → DataLoader |

**Bất biến chống rò rỉ:** mọi thống kê (clip bound, scaler min/max) đều **fit trên
train**; chia tập **trước** khi tạo cửa sổ.